# Flow con Memory: preferencias que persisten

Clasificación: **Workflow con Flow y memory.** La crew recuerda interacciones anteriores y las usa para personalizar el itinerario.

## Por qué memory aquí

Sin memory, cada ejecución empieza de cero. Si el usuario dice "no me gustan los museos" en la primera ejecución, en la segunda ya no lo sabe.

Con `memory=True` en el Crew, CrewAI activa cuatro tipos de memoria:

| Tipo | Almacenamiento | Para qué |
|------|---------------|----------|
| Short-term | ChromaDB (RAM) | Interacciones dentro de la ejecución actual. |
| Long-term | SQLite (disco) | Insights que persisten entre sesiones. |
| Entity | ChromaDB (RAM) | Personas, lugares y conceptos clave. |
| Contextual | Integra los tres | Respuestas coherentes usando todo lo anterior. |

Además, el Flow puede usar `self.remember()` y `self.recall()` para guardar/recuperar datos explícitos entre ejecuciones.

## Qué vamos a demostrar

1. Primera ejecución: el usuario indica preferencias (ej: "odio los museos, prefiero naturaleza"). Se guardan con `self.remember()` y la crew trabaja con `memory=True`.
2. Segunda ejecución: el flow recupera las preferencias con `self.recall()` y la crew recuerda patrones de ejecuciones anteriores via long-term memory. El itinerario resultante no incluye museos y prioriza naturaleza.

In [ ]:
!uv pip install -r requirements.txt --quiet

In [2]:
from dotenv import load_dotenv
import nest_asyncio

load_dotenv()
nest_asyncio.apply()

## El Flow con memory

Pasos:
1. `pedir_datos` (start): pide destino, días, personas, presupuesto y preferencias.
2. `guardar_preferencias` (listen): guarda las preferencias con `self.remember()` para futuras ejecuciones.
3. `planificar` (listen): recupera preferencias pasadas con `self.recall()`, las pasa al planificador, y lanza la crew con `memory=True` para que los agentes también recuerden.

La crew usa `memory=True`, lo que permite que los agentes acumulen contexto entre ejecuciones via long-term memory (SQLite). El `self.remember()`/`self.recall()` del Flow es una capa adicional para datos explícitos como preferencias del usuario.

In [3]:
from viajes_crew import ViajesCrew

In [4]:
from pydantic import BaseModel
from crewai import Agent, Task, Crew
from crewai.flow.flow import Flow, start, listen


class ViajeMemoryState(BaseModel):
    destino: str = ""
    dias: int = 0
    personas: int = 0
    presupuesto: int = 0
    preferencias: str = ""
    preferencias_previas: str = ""
    itinerario: str = ""


class ViajesMemoryFlow(Flow[ViajeMemoryState]):

    @start()
    def pedir_datos(self):
        print("\n=== Planificador de Viajes (con memory) ===\n")
        self.state.destino = input("Destino: ")
        self.state.dias = int(input("Días: "))
        self.state.personas = int(input("Personas: "))
        self.state.presupuesto = int(input("Presupuesto (EUR): "))
        self.state.preferencias = input("Preferencias (ej: 'odio museos, prefiero naturaleza', o Enter para ninguna): ")
        return self.state

    @listen(pedir_datos)
    def guardar_preferencias(self, state):
        """Guarda las preferencias del usuario para futuras ejecuciones."""
        if state.preferencias.strip():
            self.remember(
                f"Preferencias del viajero: {state.preferencias}",
                scope="/viajes/preferencias",
            )
            print(f"Preferencias guardadas: {state.preferencias}")
        else:
            print("Sin preferencias nuevas.")
        return state

    @listen(guardar_preferencias)
    def planificar(self, state):
        """Recupera preferencias pasadas y planifica con memory en la crew."""
        memorias = self.recall("preferencias del viajero", limit=5, depth="shallow")
        if memorias:
            previas = [m.record.content for m in memorias]
            self.state.preferencias_previas = "\n".join(previas)
            print(f"\nPreferencias recordadas de sesiones anteriores:")
            for p in previas:
                print(f"  - {p}")
        else:
            self.state.preferencias_previas = ""
            print("\nNo hay preferencias previas guardadas.")

        contexto_preferencias = ""
        if self.state.preferencias_previas:
            contexto_preferencias += f"Preferencias recordadas:\n{self.state.preferencias_previas}\n"
        if self.state.preferencias:
            contexto_preferencias += f"Preferencias nuevas: {self.state.preferencias}\n"

        planificador = Agent(
            role="Planificador de Viajes",
            goal=f"Crear un itinerario a {state.destino} respetando las preferencias del viajero",
            backstory="Diseñas viajes personalizados. Priorizas las preferencias del cliente.",
        )
        planificar_task = Task(
            description=(
                f"Crea un itinerario de {state.dias} días a {state.destino} para {state.personas} personas.\n"
                f"Presupuesto máximo: {state.presupuesto} EUR.\n"
                f"{contexto_preferencias}\n"
                "Incluye: vuelos, alojamiento, transporte, actividades dia a dia.\n"
                "Respeta las preferencias del viajero al elegir actividades."
            ),
            expected_output="Itinerario día a día con actividades adaptadas a las preferencias.",
            agent=planificador,
        )
        result = Crew(
            agents=[planificador],
            tasks=[planificar_task],
            verbose=True,
            memory=True,
        ).kickoff()
        self.state.itinerario = result.raw
        return result.raw

## Ejecución

Ejecuta esta celda dos veces:
- **Primera vez**: indica preferencias (ej: "odio museos, prefiero naturaleza y gastronomía local").
- **Segunda vez**: deja preferencias vacío (Enter). El flow recupera las anteriores y genera un itinerario coherente con ellas.

In [ ]:
flow = ViajesMemoryFlow()
result = flow.kickoff()
print(result)